# Evaluating model outputs: accuracy, precision, recall, F1, and the confusion matrix

This is the reusable evaluation template for the course. Point it at any set of **true labels** and **predicted labels** and it will tell you not just *whether the model ran*, but *whether it was right* — and, critically, *how* it was wrong.

**Time**: ~20 minutes
**Cost**: A few cents at most (we'll estimate before we run anything — see Session 1's token cost guide)

## The model ran versus the model was right

An API call can succeed — no error, a clean response, a confident-sounding answer — and still be **wrong**. "It ran" tells you the plumbing worked. It tells you nothing about whether you can trust the output for a business decision. Language models write the same way whether or not they are correct, so the smoothness of an answer carries no information about the accuracy of its content.

This is not a hypothetical risk. Zillow priced homes it was buying partly on an algorithmic valuation; when the model's errors went unmeasured against a market moving faster than expected, the company wrote down $407.9 million in inventory for 2021 and closed the business, cutting about a quarter of its workforce. Epic's sepsis-prediction model ran in hundreds of hospitals before an [independent validation study](https://jamanetwork.com/journals/jamainternalmedicine/fullarticle/2781307) at Michigan Medicine found it caught only a third of real cases. Neither failure was one wrong number — each was a wrong number reaching a decision with nothing in between to catch it.

That gap is what evaluation closes: instead of eyeballing a handful of outputs and calling it good, you compare the model's predictions against a set of **known correct answers** (called *ground truth* or *true labels*) and measure exactly how often, and in what way, it gets things wrong.

The next section in this course covers human-in-the-loop validation — where a human needs to check a model's output before it reaches a decision like the ones above. Everything below is the tool you'd use to decide where that check belongs.

## Setup

If you completed the Session 1 setup guide, your Gemini API key is already saved in Colab Secrets — there's nothing extra to do here. If you haven't, go do that first: open `setup_guide.ipynb` in the `session_1` folder of the course repository.

In [ ]:
!pip install -q google-genai

In [ ]:
from google import genai
from google.colab import userdata

# Config: change this one line to point the whole notebook at a different Gemini model.
# Swapping to a different provider (OpenAI, Anthropic, ...) means swapping this client
# and the call inside classify() below — see Session 5 for a provider-agnostic pattern.
MODEL_NAME = "gemini-2.5-flash-lite"

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
print("Connected.")

## The evaluation dataset

To keep this notebook self-contained and privacy-safe, we're using a small **synthetic** dataset — 45 invented customer feedback snippets, each hand-labeled with one of five topics an agency or CX team would recognize. No real customer data, nothing to anonymize.

**Swap this out** for your own labeled data later — anything with a text column and a true-label column works with the rest of this notebook.

In [ ]:
import pandas as pd

CATEGORIES = [
    "Shipping & Delivery",
    "Product Quality",
    "Pricing & Billing",
    "Customer Service",
    "Returns & Refunds",
]

data = [
    ("My package arrived five days later than the estimated delivery date.", "Shipping & Delivery"),
    ("Tracking said it was out for delivery but it never showed up.", "Shipping & Delivery"),
    ("Shipping was actually really fast, got it in two days!", "Shipping & Delivery"),
    ("The box arrived completely crushed, I think it was thrown around.", "Shipping & Delivery"),
    ("Delivery driver left it in the rain even though I asked for it to go behind the gate.", "Shipping & Delivery"),
    ("I paid for express shipping but it still took a week.", "Shipping & Delivery"),
    ("Great communication throughout, I always knew where my order was.", "Shipping & Delivery"),
    ("Wrong address on the label caused a huge delay.", "Shipping & Delivery"),
    ("The courier was really friendly and delivered right on time.", "Shipping & Delivery"),

    ("The fabric started pilling after just one wash.", "Product Quality"),
    ("Way better quality than I expected for the price.", "Product Quality"),
    ("The zipper broke on the second use.", "Product Quality"),
    ("Solid build, feels like it'll last for years.", "Product Quality"),
    ("Colors looked nothing like the photos online.", "Product Quality"),
    ("Stitching came undone within a week.", "Product Quality"),
    ("This is by far the best version of this product I've owned.", "Product Quality"),
    ("The material feels cheap and flimsy.", "Product Quality"),
    ("Exceeded my expectations, very well made.", "Product Quality"),

    ("I was charged twice for the same order.", "Pricing & Billing"),
    ("The price displayed at checkout didn't match what was on my card statement.", "Pricing & Billing"),
    ("Great value for the price, would buy again.", "Pricing & Billing"),
    ("Subscription renewed without any warning and charged my card.", "Pricing & Billing"),
    ("There was a hidden fee I wasn't told about until checkout.", "Pricing & Billing"),
    ("Prices are way too high compared to competitors.", "Pricing & Billing"),
    ("Refund for the price difference was processed quickly.", "Pricing & Billing"),
    ("I appreciate the transparent pricing, no surprises.", "Pricing & Billing"),
    ("My discount code didn't apply and I paid full price.", "Pricing & Billing"),

    ("The support agent was incredibly patient and solved my issue in minutes.", "Customer Service"),
    ("I waited on hold for over an hour and never got through.", "Customer Service"),
    ("Nobody responded to my email for a week.", "Customer Service"),
    ("The chat agent was rude and dismissive.", "Customer Service"),
    ("They went above and beyond to fix my problem.", "Customer Service"),
    ("I had to explain my issue three times to three different agents.", "Customer Service"),
    ("Quick, friendly, and knowledgeable support team.", "Customer Service"),
    ("Support kept transferring me in circles.", "Customer Service"),
    ("Really appreciated the follow-up call to make sure everything was resolved.", "Customer Service"),

    ("My return was processed within two days, very smooth.", "Returns & Refunds"),
    ("I've been waiting three weeks for my refund and still nothing.", "Returns & Refunds"),
    ("The return label they sent didn't work at the post office.", "Returns & Refunds"),
    ("Easiest return process I've ever dealt with.", "Returns & Refunds"),
    ("They refused to refund me even though the item was defective.", "Returns & Refunds"),
    ("Refund showed up on my card exactly as promised.", "Returns & Refunds"),
    ("Had to pay for return shipping even though it was their mistake.", "Returns & Refunds"),
    ("Customer service made the return painless.", "Returns & Refunds"),
    ("Still waiting on a refund from a return I sent back a month ago.", "Returns & Refunds"),
]

df = pd.DataFrame(data, columns=["text", "true_label"])
print(f"{len(df)} labeled examples across {df['true_label'].nunique()} categories")
df.sample(5, random_state=1)

## Generate predictions

Now let's have Gemini classify each snippet, without telling it the true label, so we have something to evaluate. This is the same pattern as item 4's "first AI assistant use case" — classification — just on our placeholder dataset instead of a live one.

**Before running a batch job, estimate the cost** (habit from the token cost guide). 45 short classification calls on `gemini-2.5-flash-lite` costs a fraction of a cent — but there's no free quota behind it anymore, so this is a good moment to check that habit even when the dollar amount is tiny.

In [ ]:
import time
from google.genai import types

CATEGORY_LIST = ", ".join(CATEGORIES)

def classify(text: str, retries: int = 2) -> str:
    prompt = f"""Classify the following customer feedback into exactly ONE of these categories:
{CATEGORY_LIST}

Feedback: "{text}"

Respond with only the category name, exactly as written above. Nothing else."""

    for attempt in range(retries + 1):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                # temperature=0 makes the model as deterministic as it can be, so re-running
                # this notebook gives you (close to) the same predictions and metrics each time.
                config=types.GenerateContentConfig(temperature=0),
            )
            label = response.text.strip()
            # Guard against the model returning something slightly off-format
            return label if label in CATEGORIES else "UNKNOWN"
        except Exception as e:
            if attempt == retries:
                print(f"  Giving up on one row after {retries} retries ({e}) — recording as UNKNOWN.")
                return "UNKNOWN"
            time.sleep(2)

predictions = [classify(text) for text in df["text"]]
df["predicted_label"] = predictions

# UNKNOWN covers both off-format replies and calls that failed every retry. It's a real
# outcome the model produced, so it's kept as its own label below (rather than silently
# dropped) -- otherwise accuracy and the confusion matrix would disagree with each other.
LABELS = CATEGORIES + ["UNKNOWN"] if "UNKNOWN" in df["predicted_label"].values else CATEGORIES

unknown_count = (df["predicted_label"] == "UNKNOWN").sum()
if unknown_count:
    print(f"⚠️ {unknown_count} response(s) didn't match a known category exactly (or failed every retry) — worth showing the class as a real-world formatting failure mode.")
print("Done.")
df.head()

## Accuracy

**Accuracy** answers one question: *of everything the model labeled, what percentage did it get exactly right?*

It's the simplest metric — and the easiest to misread. A model can score 90% accuracy and still be useless if the 10% it gets wrong are the cases that matter most to your business (e.g. always missing the angriest, highest-churn-risk complaints).

Picture a lead-scoring model instead: 10,000 people fill in a contact form each month, and about 500 of them — 5% — are serious buyers. A model that flags nobody at all, answering "not a buyer" every single time, is right 95% of the time without having learned anything. A real model that flags 1,200 submissions and gets 300 of them right, correctly leaving the other 8,600 alone, scores (300 + 8,600) / 10,000 = **89%** — lower than the model that does nothing.

Before you report an accuracy figure, work out what you would score by always guessing the most common category. If your model isn't clearly ahead of that baseline, you haven't shown anything yet. The same trap applies to this notebook's own dataset, which is deliberately balanced at nine items per category so it can't demonstrate the effect — real customer feedback is almost never evenly split, and that's where this matters.

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(df["true_label"], df["predicted_label"])
print(f"Overall accuracy: {accuracy:.1%}  ({int(accuracy * len(df))} of {len(df)} correct)")

## The confusion matrix

Accuracy gives you one number. The **confusion matrix** shows you *where the model gets confused* — which categories it mixes up with which. Rows are the true label, columns are what the model predicted; the diagonal is everything it got right, and everything off the diagonal is a specific kind of mistake.

Reading across a row shows where a category's items actually went, which tells you what that category is losing. Reading down a column shows what got pulled into a category, which tells you what it's over-collecting.

Which mistakes a model makes usually costs more or less than how many it makes. An item drifting between Pricing & Billing and Customer Service barely matters if the same team triages both queues. An item drifting from Returns & Refunds into Customer Service is expensive if a refund mention is what triggers a churn-prevention workflow — each one is a customer who dropped out of a process built to keep them, and nothing in the accuracy figure will tell you it happened. Accuracy tells you the model was wrong 10 times out of 100; the matrix tells you which 10, and that's the piece of information you can actually act on.

One note on axes: scikit-learn's `confusion_matrix`, used below, puts true labels on the rows and predictions on the columns, matching the description above. Some other tools flip this, so check the axis labels before reading anyone else's matrix.

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(df["true_label"], df["predicted_label"], labels=LABELS)

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=LABELS, yticklabels=LABELS)
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.title('Confusion Matrix')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Precision, recall, and F1, by category

Accuracy treats every category the same. These three metrics look at each category individually, and answer three different questions:

- **Precision** — *of everything the model labeled as this category, how much actually was?* Precision = true positives / (true positives + **false positives**). A **false positive** is a row the model labeled as this category that actually belongs to a different one — the confusion matrix column for this category, minus its diagonal cell. Low precision means the model cries wolf — it over-labels this category.
- **Recall** — *of everything that actually was this category, how much did the model catch?* Recall = true positives / (true positives + **false negatives**). A **false negative** is a row that actually belongs to this category but the model labeled as something else — the confusion matrix row for this category, minus its diagonal cell. Low recall means the model misses real cases — it under-labels this category.
- **F1 score** — the harmonic mean of precision and recall: `2 × (precision × recall) / (precision + recall)`. Not a plain average — it drops toward whichever of the two is lower, instead of splitting the difference. Useful when you want one number per category instead of two. F1 weights precision and recall equally, which is itself an assumption — it says the two error types cost the same. When they don't, weighted variants exist: F2 leans toward recall, F0.5 toward precision.

Picking a category to measure means picking a **threshold** too — the point above which the model's output counts as a positive prediction for that category. Lower it and the model flags more, so recall rises (fewer real cases slip past) while precision falls (more of what got flagged shouldn't have been); raise it and the reverse happens. No setting maximises both, so choosing one means choosing which mistake you'd rather make — a business decision, not a statistical one.

For Returns & Refunds, missing a genuine return request (low recall, a false negative) usually costs more than occasionally flagging something that wasn't one (low precision, a false positive): the miss is an unresolved customer, and the false alarm is a few seconds of an agent's attention. For a category that feeds an automated action instead of a human queue, that calculus can flip — a false positive that reaches a customer unprompted is a different kind of expensive.

Once precision gets low enough on a high-volume category, people stop working the flagged list at all, and a list nobody works has an effective recall of zero regardless of what the model is doing — a failure mode worth naming: **alert fatigue**. A low-precision model can destroy the human check that was supposed to protect against it, which is exactly why high-stakes categories need a human review step sized to the category's actual error costs, not to the model's overall accuracy.

One vocabulary note for reading outside marketing: precision is what medicine and statistics call **positive predictive value (PPV)**; recall is **sensitivity**. There's also **specificity** — true negatives / (true negatives + false positives) — which has no standard machine-learning name of its own. Same quantities, different fields.

In [ ]:
from sklearn.metrics import classification_report

# zero_division=0 controls what to print when a category never appears in the predictions
# (or never appears in the true labels) -- without it, sklearn raises a warning for the
# resulting 0/0 division and asks you to pick a value; 0 is the standard "undefined" choice.
print(classification_report(df["true_label"], df["predicted_label"], labels=LABELS, zero_division=0))

### Where F1 actually comes from

`classification_report` prints F1, but it's worth seeing the arithmetic behind it at least once. For any one category, F1 is the harmonic mean of that category's precision and recall — not the plain average — which is why it drops sharply if either precision or recall is low, rather than just splitting the difference between them. A model that flags every single item scores 100% recall and near-zero precision; a plain average of the two would still look presentable, while the harmonic mean collapses toward zero — the right answer, since a model that flags everything has told you nothing.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(
    df["true_label"], df["predicted_label"], labels=LABELS, zero_division=0
)

# Show the arithmetic for one category, and confirm it matches sklearn's F1 above
example_idx = 0
p, r = precision[example_idx], recall[example_idx]
manual_f1 = 2 * (p * r) / (p + r) if (p + r) else 0.0

print(f"Category: {LABELS[example_idx]}")
print(f"  precision = {p:.3f}, recall = {r:.3f}")
print(f"  harmonic mean  2 * (precision * recall) / (precision + recall) = {manual_f1:.3f}")
print(f"  sklearn's F1                                                   = {f1[example_idx]:.3f}")

## The reusable template

Everything above, wrapped into one function. This is the part anyone in the course can copy into their own notebook: swap in your own `true_label` / `predicted_label` columns and run it.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

def evaluate(y_true, y_pred, labels=None, title="Evaluation Results"):
    """
    Evaluate classification predictions against ground truth.
    Works on ANY labeled classification task -- not just this dataset.

    y_true, y_pred : lists or pandas Series of labels (same length, same order)
    labels         : ordered list of all possible category names (optional --
                      inferred from the data if not given)
    """
    observed = set(y_true) | set(y_pred)
    if labels is None:
        labels = sorted(observed)
    else:
        # Keep any label the caller passed in, but don't silently drop a label that shows
        # up in the data and wasn't listed (e.g. an UNKNOWN/fallback value) -- dropping it
        # would make accuracy and the confusion matrix disagree on how many rows there are.
        labels = list(labels) + sorted(observed - set(labels))

    accuracy = accuracy_score(y_true, y_pred)
    print(f"{title}")
    print("=" * len(title))
    print(f"Overall accuracy: {accuracy:.1%}\n")

    print("Per-category breakdown (precision / recall / F1):")
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted label')
    plt.ylabel('True label')
    plt.title('Confusion Matrix')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    return {"accuracy": accuracy, "confusion_matrix": cm}

# Try it on our dataset:
results = evaluate(df["true_label"], df["predicted_label"], labels=CATEGORIES, title="Topic Classification Evaluation")

## Using this on your own data

To evaluate your own model outputs instead of this placeholder dataset:

1. Get your data into a table with (at minimum) a `true_label` column and a `predicted_label` column — same row order, same category names in both.
2. If you're starting from a CSV: `df = pd.read_csv('your_file.csv')`
3. Run: `evaluate(df['true_label'], df['predicted_label'], labels=[...your categories...])`

That's it — the function doesn't care what the categories are or where the predictions came from (Gemini, another model, or even a human).

## Wrap-up

You now have a working, reusable way to answer "was the model actually right?" instead of just "did it run?" — accuracy for the overall picture, the confusion matrix for *where* it gets confused, and precision/recall/F1 for *how* it's wrong on each category.

**Next in Session 2**: Human-in-the-loop validation — at what point does a wrong prediction actually reach a business decision, and where does a human need to check it first? The per-category breakdown above is exactly the tool you'd use to decide that.

**Questions?** Post in the Circle community.